### Задание 1 (8 баллов + 0.5)

Скачайте вот этот датасет - ("MERA-evaluation/MERA", "multiq"). Так как обучать ничего не придется возьмите train split. Возьмите оттуда 25 примеров (случайных, а не первых). 

Преобразуйте колонку instruction, заменив переменные вроде `{support_text}` на реальные значения из других колонок, чтобы получился готовый промпт (инпут для модели).

#### Метрика (2 балла)

Напишите две метрики для оценки правильных ответов: 1) exact match и 2) F1 по леммам. Для первой просто добавьте базовую предобработку вроде удаления пунктуации и регистра и просто сопоставьте строки между собой. Для второй к базовой предобработке добавьте токенизацию и лемматизацию, а поверх лемм рассчитайте F1 метрику (подсказка - precision можно рассчитать посчитав какой процент лемм в предсказанном ответе совпадает с правильным ответом). 

Рассчитайте метрики на простом бейзлайне (например, можете попробовать взять самый частотный биграмм из промпта в качестве ответа). Можно получить +0.5 балла за неочевидный ненулевой бейзлайн.

#### Эксперимент 1 (2 балла)
Возьмите какую-нибудь не очень большую модель с huggingface (например, `Qwen/Qwen3.5-0.8B`, но можно и любую другую, которая умеет отвечать на инструкции и поддерживает thinking). Преобразуйте промпты из датасета в формат `messages = [{"role": "user", "content": prompt}]` с помощью чат темплейта этой модели. 
Сгенерируйте ответы модели на полученных данных (25 примеров). На этом шаге не используйте thinking. Поставьте max_new_tokens на не очень большое значения, чтобы генерация не занимала слишком много времени. 
Оцените качество ответов с помощью двух метрик.

#### Эксперимент 2 (1 балла)
К первому эксперименту добавьте какой-то system prompt, который по вашему должен помочь модели справится с результатом. Можете проанализировать ошибки из первого задания и написать инструкции на основе этого. 
Сгенерируйте ответы с точно такими же параметрами как и в эксперименте 1. 
Оцените качество ответов с помощью двух метрик.

#### Эсперимент 3 (3 балла)
На этом шаге включите thinking в модели и сгенерируйте предсказания (system prompt вы можете использовать по вашему желанию).
На этом шаге у вас будет несколько сложностей, которые нужно решить: 
1) генерацию модели нельзя использовать напрямую, сначала нужно отделить thinking блок от самого ответа. Добавьте пост обработку
2) thinking может занять достаточно много токенов и его нужно ограничить, но при этом вам нужно, чтобы модель выдала и сам ответ, а не только незаконченные размышлений. Выберите max_new_tokens с рассчетом на (thinking_tokens + answer_tokens). Сначала запустите первую генерацию с thinking_tokens, затем проверьте, что в ответе содержится </think> тэг, который означает, что модель закончила думать (еще удобно использовать `stop_strings` в huggingface generation параметрах, чтобы заканчивать как только появился нужный токен). И если тэг есть, то просто запустите вторую генерацию с max_new_tokens (как продолжение уже накопившегося) или же поставьте </think> тэг вручную (обрубив размышление) и запустите вторую генерацию c answer_tokens. 


In [1]:
!pip install datasets transformers accelerate pymorphy3 torch


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import ast
import random
import re
import string
from datasets import load_dataset

ds = load_dataset("ai-forever/MERA", "multiq", split="train")

random.seed(42)
indices = random.sample(range(len(ds)), 25)
sample = ds.select(indices)


def build_prompt(example):
    instruction = example["instruction"]
    inputs = example["inputs"]
    return instruction.format(
        support_text=inputs.get("support_text", ""),
        text=inputs.get("text", ""),
        question=inputs.get("question", ""),
    )


def extract_gold(out):
    if isinstance(out, list):
        out = out[0] if out else ""
    if isinstance(out, str):
        try:
            d = ast.literal_eval(out)
            if isinstance(d, dict):
                return d.get("segment", "").strip()
        except (ValueError, SyntaxError):
            pass
        return out.strip()
    if isinstance(out, dict):
        return out.get("segment", "").strip()
    return ""


prompts = [build_prompt(ex) for ex in sample]
gold_answers = [extract_gold(ex["outputs"]) for ex in sample]

print(f"Загружено примеров: {len(prompts)}")
for i in range(3):
    print(f"Gold[{i}]: {gold_answers[i]!r}")
print("\nПример")
print(prompts[0][:500])

c:\Users\1\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Загружено примеров: 25
Gold[0]: 'Таз'
Gold[1]: 'Вольный ганзейский город Бремен'
Gold[2]: 'городским советом'

Пример
Даны два текста:
Текст 1: Ньюгантокылькы ("устар." Ньюган-Токаль-Кы) — река в России, протекает по территории Ямало-Ненецкого автономного округа. Устье реки находится в 180 км по левому берегу реки Толька. Длина реки составляет 58 км. В 12 км от устья по левому берегу реки впадает река Кутылькы.
Текст 2: Толька — река в России, протекает в Ямало-Ненецком автономном округе. Впадает в реку Таз слева на 723-м километре от устья. Длина реки — 391 км, площадь водосборного бассейна — 13 300 кв км. Сре


In [3]:
import re
import string
from collections import Counter
import pymorphy3

morph = pymorphy3.MorphAnalyzer()


def normalize_text(text: str) -> str:
    text = (text or "").lower()
    text = text.translate(str.maketrans("", "", string.punctuation + "«»—–"))
    text = re.sub(r"\s+", " ", text).strip()
    return text


def exact_match(pred: str, gold: str) -> float:
    return 1.0 if normalize_text(pred) == normalize_text(gold) else 0.0


def lemmatize(text: str) -> list:
    return [morph.parse(tok)[0].normal_form for tok in normalize_text(text).split()]


def f1_lemmas(pred: str, gold: str) -> float:
    pl, gl = lemmatize(pred), lemmatize(gold)
    if not pl or not gl:
        return 0.0
    overlap = sum((Counter(pl) & Counter(gl)).values())
    if overlap == 0:
        return 0.0
    p = overlap / len(pl)
    r = overlap / len(gl)
    return 2 * p * r / (p + r)


def evaluate(predictions, golds):
    em = [exact_match(p, g) for p, g in zip(predictions, golds)]
    f1 = [f1_lemmas(p, g) for p, g in zip(predictions, golds)]
    return {"EM": sum(em) / len(em), "F1": sum(f1) / len(f1)}


print("EM check:", exact_match("Таз", "таз"))            # 1.0
print("F1 check:", f1_lemmas("река Таз", "Таз"))         # ≈0.67

EM check: 1.0
F1 check: 0.6666666666666666


In [4]:
STOPWORDS = {"текст", "ответ", "вопрос", "данные", "опираясь", "запишите"}


def extract_capitalized_phrases(text: str) -> list:
    words = re.findall(r"[А-ЯЁA-Z][А-Яа-яЁёA-Za-z\-]+", text)
    phrases, buf = [], []
    for w in words:
        if w.lower() in STOPWORDS:
            if buf:
                phrases.append(" ".join(buf))
                buf = []
            continue
        buf.append(w)
    if buf:
        phrases.append(" ".join(buf))
    return [p for p in phrases if len(p) >= 2]


def _context_before_question(prompt: str) -> str:
    for marker in ["Опираясь на данные тексты", "Вопрос:", "Ответ:"]:
        if marker in prompt:
            return prompt.split(marker)[0]
    return prompt


def baseline_most_frequent_entity(prompts, golds):
    preds = []
    for prompt in prompts:
        phrases = extract_capitalized_phrases(_context_before_question(prompt))
        if not phrases:
            preds.append("")
            continue
        counter = Counter(phrases)
        top = sorted(counter.items(), key=lambda kv: (-kv[1], -len(kv[0])))
        preds.append(top[0][0])
    return preds


def baseline_longest_entity(prompts, golds):
    preds = []
    for prompt in prompts:
        phrases = extract_capitalized_phrases(_context_before_question(prompt))
        if not phrases:
            preds.append("")
            continue
        counter = Counter(phrases)
        top = sorted(counter.items(), key=lambda kv: (-len(kv[0]), -kv[1]))
        preds.append(top[0][0])
    return preds


baseline1_preds = baseline_most_frequent_entity(prompts, gold_answers)
baseline1_metrics = evaluate(baseline1_preds, gold_answers)
print("Baseline 1 (most frequent capitalized phrase):")
print(f"  EM: {baseline1_metrics['EM']:.4f}")
print(f"  F1: {baseline1_metrics['F1']:.4f}")

baseline2_preds = baseline_longest_entity(prompts, gold_answers)
baseline2_metrics = evaluate(baseline2_preds, gold_answers)
print("\nBaseline 2 (longest capitalized phrase):")
print(f"  EM: {baseline2_metrics['EM']:.4f}")
print(f"  F1: {baseline2_metrics['F1']:.4f}")

if baseline2_metrics["F1"] > baseline1_metrics["F1"]:
    baseline_preds, baseline_metrics = baseline2_preds, baseline2_metrics
    baseline_name = "longest capitalized phrase"
else:
    baseline_preds, baseline_metrics = baseline1_preds, baseline1_metrics
    baseline_name = "most frequent capitalized phrase"

print(f"\nВыбран основной baseline: {baseline_name}")
print(f"  EM: {baseline_metrics['EM']:.4f}")
print(f"  F1: {baseline_metrics['F1']:.4f}")

Baseline 1 (most frequent capitalized phrase):
  EM: 0.0000
  F1: 0.0837

Baseline 2 (longest capitalized phrase):
  EM: 0.0000
  F1: 0.0837

Выбран основной baseline: most frequent capitalized phrase
  EM: 0.0000
  F1: 0.0837


In [5]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
torch.set_num_threads(4)   # Windows: без этой строки PyTorch «залипает»

MODEL_NAME = "Qwen/Qwen3-0.6B"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,   # CPU-совместимо, без offload
    device_map=None,             # НЕ "auto"
)
model = model.to("cpu")
model.eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id

print("Модель:", MODEL_NAME)
print("Устройство:", model.device)
print("pad_token_id:", tokenizer.pad_token_id)
print("Потоков CPU:", torch.get_num_threads())

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 311/311 [00:02<00:00, 151.23it/s]


Модель: Qwen/Qwen3-0.6B
Устройство: cpu
pad_token_id: 151643
Потоков CPU: 4


In [7]:
import re
import torch


def postprocess_answer(text: str) -> str:
    """Компактный ответ: срезает 'Ответ:', берёт первую непустую строку"""
    text = text.strip()
    m = re.findall(r"(?:Ответ|Answer)\s*[:\-]\s*(.+)", text, flags=re.IGNORECASE)
    if m:
        text = m[-1]
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if lines:
        text = lines[0]
    return text.strip().strip('"\'«»').rstrip(".")


def generate_answers(prompts, system_prompt=None, enable_thinking=False,
                     max_new_tokens=24, verbose=True, tag="gen"):
    """Обычная генерация (один проход)"""
    answers, raws = [], []
    for i, prompt in enumerate(prompts):
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": prompt})

        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=enable_thinking,
        )
        enc = tokenizer([text], return_tensors="pt").to(model.device)
        prompt_len = enc.input_ids.shape[1]

        with torch.no_grad():
            out = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                min_new_tokens=1,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        new_ids = out[0][prompt_len:].tolist()
        raw = tokenizer.decode(new_ids, skip_special_tokens=True)
        raws.append(raw)
        answers.append(postprocess_answer(raw))
        if verbose:
            print(f"[{tag} {i+1}/{len(prompts)}] {answers[-1][:60]!r}", flush=True)
    return answers, raws


print("Функции готовы.")

Функции готовы.


In [8]:
preds_exp1, raws_exp1 = generate_answers(
    prompts, system_prompt=None, enable_thinking=False,
    max_new_tokens=24, verbose=True, tag="exp1",
)
metrics_exp1 = evaluate(preds_exp1, gold_answers)

print("\nЭксперимент 1 (no system, no thinking):")
print(f"  EM: {metrics_exp1['EM']:.4f}")
print(f"  F1: {metrics_exp1['F1']:.4f}")
print(f"  Непустых: {sum(1 for p in preds_exp1 if p.strip())}/{len(preds_exp1)}")

for i in range(5):
    print(f"\nпример {i}")
    print(f"Raw:   {raws_exp1[i]!r}")
    print(f"Pred:  {preds_exp1[i]!r}")
    print(f"Gold:  {gold_answers[i]!r}")

[exp1 1/25] 'Ньюгантокыльк'
[exp1 2/25] 'Бремен'
[exp1 3/25] 'городской совет'
[exp1 4/25] 'административно-территориальная единица — **Литва**'
[exp1 5/25] 'Река, притоком которой является Морочь, впадает в **Случь**'
[exp1 6/25] 'Шевчук, Виктор Иванович'
[exp1 7/25] 'Слюняев Игорь Николаевич'
[exp1 8/25] 'Казахстан'
[exp1 9/25] 'Где родился человек, который написал музыку к фильму "Тайна '
[exp1 10/25] 'Иртыш'
[exp1 11/25] 'Кардинал-священник с 25 февраля 1965'
[exp1 12/25] 'Эне-Термотъега сливается рекой Пывъях'
[exp1 13/25] 'Большой Юган'
[exp1 14/25] 'Карлос де лос Кобос играет в **Лиге МХ**'
[exp1 15/25] 'Михаил Викторович Желудов'
[exp1 16/25] 'Император'
[exp1 17/25] 'Малый Ямсовей'
[exp1 18/25] 'Законодательный орган города, где умер Юрий Александрович Де'
[exp1 19/25] 'Худой-Ёль сливается с рекой Печора'
[exp1 20/25] 'Аманулла Габдулхаевич Рамазанов родился в Павл'
[exp1 21/25] 'Река, питающаяся от реки, притоком которой является **Хоханъ'
[exp1 22/25] 'Река, притоком которой

In [9]:

SYSTEM_PROMPT = (
    "Ты — эксперт по вопросно-ответным задачам на русском языке. "
    "Тебе даны два текста и вопрос. Найди ответ, используя информацию из обоих текстов. "
    "Отвечай только ответом: без объяснений, без вводных слов, без кавычек, "
    "без префикса 'Ответ:'. Если ответ это имя собственное или название, пиши его в той форме, в которой оно встречается в тексте. "
    "Не добавляй пояснений и не пересказывай текст."
)

preds_exp2, raws_exp2 = generate_answers(
    prompts, system_prompt=SYSTEM_PROMPT, enable_thinking=False,
    max_new_tokens=24, verbose=True, tag="exp2",
)
metrics_exp2 = evaluate(preds_exp2, gold_answers)

print("\nЭксперимент 2 (system prompt, no thinking):")
print(f"  EM: {metrics_exp2['EM']:.4f}")
print(f"  F1: {metrics_exp2['F1']:.4f}")
print(f"  Непустых: {sum(1 for p in preds_exp2 if p.strip())}/{len(preds_exp2)}")

for i in range(5):
    print(f"\nпример {i}")
    print(f"Raw:   {raws_exp2[i]!r}")
    print(f"Pred:  {preds_exp2[i]!r}")
    print(f"Gold:  {gold_answers[i]!r}")

[exp2 1/25] 'Ньюгантокылькы'
[exp2 2/25] 'Бремен'
[exp2 3/25] 'Ереван'
[exp2 4/25] 'Иван Иванович Луцкевич'
[exp2 5/25] 'Случь'
[exp2 6/25] 'Шапа, Пётр Иванович'
[exp2 7/25] 'Слюняев Игорь Николаевич'
[exp2 8/25] 'Казахстан'
[exp2 9/25] 'Новосибирск'
[exp2 10/25] 'Бича'
[exp2 11/25] 'парламент'
[exp2 12/25] 'Эне-Термотъега'
[exp2 13/25] 'Большой Юган'
[exp2 14/25] 'Карлос Де лос Кобос играет в Лиге МХ'
[exp2 15/25] 'Михаил Викторович Желудов'
[exp2 16/25] 'Император'
[exp2 17/25] 'Малый Ямсовей'
[exp2 18/25] 'Нижегородский кремль'
[exp2 19/25] 'Печора'
[exp2 20/25] 'Аманулла Габдулхаевич Рамазанов'
[exp2 21/25] 'Река, питающаяся от реки, приток которой является Хоханъя, —'
[exp2 22/25] 'Катка'
[exp2 23/25] 'Мэр Казань'
[exp2 24/25] 'городской совет'
[exp2 25/25] 'Бох-Джага'

Эксперимент 2 (system prompt, no thinking):
  EM: 0.1200
  F1: 0.2507
  Непустых: 25/25

пример 0
Raw:   'Ньюгантокылькы.'
Pred:  'Ньюгантокылькы'
Gold:  'Таз'

пример 1
Raw:   'Бремен'
Pred:  'Бремен'
Gold:  'Воль

In [10]:
import re
import torch


def generate_with_thinking(prompts, system_prompt=None,
                           thinking_tokens=32, answer_tokens=24,
                           verbose=True):
    """
    Шаг 1: генерируем «размышление» до </think> (или до лимита).
    Шаг 2: генерируем ответ, начиная с накопленного префикса.
    Если </think> не появился, то обрубаем и добавляем вручную.
    """
    answers, thinkings = [], []
    for i, prompt in enumerate(prompts):
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": prompt})

        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=True,
        )
        enc = tokenizer([text], return_tensors="pt").to(model.device)
        prompt_len = enc.input_ids.shape[1]

        # --- Шаг 1: thinking ---
        with torch.no_grad():
            think_ids = model.generate(
                **enc,
                max_new_tokens=thinking_tokens,
                min_new_tokens=1,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                stop_strings=["</think>"],
                tokenizer=tokenizer,
            )
        think_text = tokenizer.decode(
            think_ids[0][prompt_len:], skip_special_tokens=False
        )
        thinkings.append(think_text)

        # --- Шаг 2: ответ ---
        if "</think>" in think_text:
            prefix = text + think_text.split("</think>")[0] + "</think>\n"
        else:
            prefix = text + think_text + "\n</think>\n"

        prefix_enc = tokenizer([prefix], return_tensors="pt").to(model.device)
        prefix_len = prefix_enc.input_ids.shape[1]

        with torch.no_grad():
            ans_ids = model.generate(
                **prefix_enc,
                max_new_tokens=answer_tokens,
                min_new_tokens=1,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        answer_text = tokenizer.decode(
            ans_ids[0][prefix_len:], skip_special_tokens=True
        )
        answers.append(postprocess_answer(answer_text))
        if verbose:
            print(f"[exp3 {i+1}/{len(prompts)}] "
                  f"think={len(think_text)}c, pred={answers[-1][:50]!r}",
                  flush=True)
    return answers, thinkings


preds_exp3, thinkings_exp3 = generate_with_thinking(
    prompts, system_prompt=SYSTEM_PROMPT,
    thinking_tokens=32, answer_tokens=24, verbose=True,
)
metrics_exp3 = evaluate(preds_exp3, gold_answers)

print("\nЭксперимент 3:")
print(f"  EM: {metrics_exp3['EM']:.4f}")
print(f"  F1: {metrics_exp3['F1']:.4f}")
print(f"  Непустых: {sum(1 for p in preds_exp3 if p.strip())}/{len(preds_exp3)}")

for i in range(5):
    print(f"\nпример {i}")
    print(f"Thinking (первые 150): {thinkings_exp3[i][:150]!r}...")
    print(f"Pred:  {preds_exp3[i]!r}")
    print(f"Gold:  {gold_answers[i]!r}")

[exp3 1/25] think=82c, pred='Ньюгантокылькы'
[exp3 2/25] think=100c, pred='Бремен'
[exp3 3/25] think=108c, pred='городской совет'
[exp3 4/25] think=95c, pred='Шавли'
[exp3 5/25] think=90c, pred='Случь'
[exp3 6/25] think=109c, pred='Шевчук'
[exp3 7/25] think=105c, pred='Ситников Сергей Константинович'
[exp3 8/25] think=107c, pred='Казахстан'
[exp3 9/25] think=97c, pred='Новосибирск'
[exp3 10/25] think=95c, pred='Бича'
[exp3 11/25] think=111c, pred='парламент'
[exp3 12/25] think=81c, pred='Пывъях'
[exp3 13/25] think=98c, pred='Большой Юган'
[exp3 14/25] think=92c, pred='Керетаро'
[exp3 15/25] think=113c, pred='Михаил Викторович Желудов'
[exp3 16/25] think=101c, pred='Император'
[exp3 17/25] think=52c, pred='Малый Ямсовей'
[exp3 18/25] think=103c, pred='Нижегородский кремль'
[exp3 19/25] think=52c, pred='Худой-Ёль сливается с рекой Печора'
[exp3 20/25] think=109c, pred='Аманулла Габдулхаевич Рамазанов'
[exp3 21/25] think=91c, pred='Хоханъя'
[exp3 22/25] think=96c, pred='Маравка'
[exp3 23/

In [11]:
import pandas as pd

rows = []

if isinstance(globals().get("baseline_metrics"), dict):
    base_name = globals().get("baseline_name", "baseline")
    rows.append((f"Baseline ({base_name})",
                 baseline_metrics.get("EM"),
                 baseline_metrics.get("F1")))

for label, var in [
    ("Exp 1",      "metrics_exp1"),
    ("Exp 2",  "metrics_exp2"),
    ("Exp 3",    "metrics_exp3"),
]:
    m = globals().get(var)
    if isinstance(m, dict):
        rows.append((label, m.get("EM"), m.get("F1")))

results = pd.DataFrame(rows, columns=["Эксперимент", "EM", "F1"])
results["EM"] = pd.to_numeric(results["EM"], errors="coerce").astype("Float64").round(4)
results["F1"] = pd.to_numeric(results["F1"], errors="coerce").astype("Float64").round(4)

print(results.to_string(index=False))
results

                                Эксперимент    EM      F1
Baseline (most frequent capitalized phrase)   0.0  0.0837
                                      Exp 1  0.12  0.2187
                                      Exp 2  0.12  0.2507
                                      Exp 3  0.16  0.3347


,Эксперимент,EM,F1
0,Baseline (most frequent capitalized phrase),0.0,0.0837
1,Exp 1,0.12,0.2187
2,Exp 2,0.12,0.2507
3,Exp 3,0.16,0.3347
